# RS HW2 — Q2 NeuMF (PyTorch)

**Reference:** He et al., *Neural Collaborative Filtering* (WWW 2017), [official author implementation](https://github.com/hexiangnan/neural_collaborative_filtering/blob/master/NeuMF.py). This is an original **PyTorch reimplementation** of its GMF + MLP + prediction-layer architecture (without GMF/MLP pretraining). The official legacy source uses Keras/Theano.

**Comparison with Q1 BPR v2:** identical temporal 80:20 split, train-only user-wise Z-scores (`Z >= 0.5`), fixed `rating >= 4` validation relevance, full-catalog NDCG@20, train-observed item masking. **NeuMF uses pointwise BCE-with-logits and multiple sampled negatives**, unlike BPR's pairwise ranking loss. Item bias is an optional **extension**, not the unmodified canonical NeuMF architecture.

**Important:** Kaggle's exact hidden relevance rule is not confirmed; local NDCG is a model-selection proxy, *not* the official score. Do not tune to the Kaggle public leaderboard repeatedly.

In [51]:
from pathlib import Path
import zipfile, random, copy, math
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Official Kaggle competition first; also supports a user-uploaded Kaggle dataset and local ZIP.
locations = [
    Path('/kaggle/input/competitions/skku-esw-4024-recommender-system'),
    Path('/kaggle/input/datasets/chaihyunlee/recsys-hw2-dataset'),
    Path('data'),
]
DATA_DIR = next((p for p in locations if all((p / f).exists() for f in ('train.csv','test.csv','sample_submission.csv'))), None)
if DATA_DIR is None:
    zip_candidates = [Path('skku-esw-4024-recommender-system.zip'), Path('/mnt/data/skku-esw-4024-recommender-system.zip')]
    archive = next((p for p in zip_candidates if p.exists()), None)
    if archive is None:
        raise FileNotFoundError('Attach the Kaggle competition dataset or place the original ZIP in the notebook directory.')
    DATA_DIR = Path('data')
    DATA_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for fn in ('train.csv','test.csv','sample_submission.csv'):
            (DATA_DIR / fn).write_bytes(z.read(fn))

OUTPUT_DIR = Path('/kaggle/working/outputs') if Path('/kaggle/working').exists() else Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_PATH, TEST_PATH, SAMPLE_PATH = [DATA_DIR / fn for fn in ('train.csv','test.csv','sample_submission.csv')]

# ===== Editable configuration =====
EXP_NAME = 'exp3'
K = 20
VAL_RATIO = 0.20
VAL_RELEVANT_RATING = 4.0         # fixed validation target across experiments
Z_THRESHOLD = 0.5                # positive training sample, train-only per-user Z-score
GMF_DIM = 32
MLP_EMBED_DIM = 32                # concatenated GMF/MLP inputs: 64 -> 32 -> 16 -> 8
MLP_HIDDEN = (32,16,8)
USE_ITEM_BIAS = True              # EXTENSION: compare False for canonical NeuMF
NEG_SAMPLING = 'popularity_mixed'          # 'uniform' or 'popularity_mixed'
POPULARITY_ALPHA = 0.5
POPULARITY_BETA = 0.75
NUM_NEGATIVES = 4                 # 4 negative samples per positive interaction
LR = 0.001
WEIGHT_DECAY = 1e-5
BATCH_SIZE = 2048
MAX_EPOCHS = 60
PATIENCE = 8
EVAL_EVERY = 1
EVAL_USER_BATCH = 8
EVAL_ITEM_BATCH = 2048
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Data:', DATA_DIR, 'Device:', DEVICE, 'Output:', OUTPUT_DIR)


Data: /kaggle/input/competitions/skku-esw-4024-recommender-system Device: cpu Output: /kaggle/working/outputs


## 1. Load datasets and verify submission format

In [52]:
train_raw = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)
assert {'userId','movieId','rating','time_idx'}.issubset(train_raw.columns)
assert {'id','userId'}.issubset(test.columns)
assert {'id','recommendations'}.issubset(sample.columns)
assert not train_raw.duplicated(['userId','movieId']).any()
assert set(test.userId).issubset(set(train_raw.userId)), 'Test contains unseen users'
assert sample.id.tolist() == test.id.tolist()
print('Train:', train_raw.shape, '| Test:', test.shape)
print('Users:', train_raw.userId.nunique(), '| Items:', train_raw.movieId.nunique())
display(train_raw.head(),test.head(),sample.head(2))

Train: (80621, 4) | Test: (588, 2)
Users: 610 | Items: 8237


,userId,movieId,rating,time_idx
0,0,3086,3.0,0
1,0,5849,2.0,1
2,0,3134,3.0,2
3,0,1948,5.0,3
4,0,383,4.0,4


,id,userId
0,0,0
1,1,1
2,2,2
3,3,3
4,4,4


,id,recommendations
0,0,4830 2814 6872 1040 6398 5783 9472 7593 5677 3...
1,1,2573 7753 4324 8635 7256 5889 9597 4762 5233 2...


## 2. User-wise temporal 80:20 split (identical to Q1 BPR v2)

In [53]:
def temporal_split(df, val_ratio=0.2):
    ordered = df.sort_values(['userId','time_idx'], kind='stable')
    groups = ordered.groupby('userId', sort=False)
    counts = groups.userId.transform('size')
    positions = groups.cumcount()
    cutoffs = np.maximum(1, np.floor(counts * (1.0-val_ratio)).astype(int))
    return (ordered.loc[positions < cutoffs].copy().reset_index(drop=True),
            ordered.loc[positions >= cutoffs].copy().reset_index(drop=True))

train_part, val_part = temporal_split(train_raw, VAL_RATIO)
val_relevant = val_part.loc[
    val_part.rating.ge(VAL_RELEVANT_RATING) & val_part.userId.isin(test.userId),
    ['userId','movieId']
].copy()
assert not set(zip(train_part.userId,train_part.movieId)) & set(zip(val_part.userId,val_part.movieId))
print('Train:',len(train_part),'Val:',len(val_part),'Val relevant:',len(val_relevant),
      'Evaluation users:',val_relevant.userId.nunique())

Train: 64246 Val: 16375 Val relevant: 7400 Evaluation users: 575


## 3. ID mapping, train-only Z-score normalization, observed/positive sets

In [54]:
user_ids = pd.Index(pd.unique(pd.concat([train_raw.userId,test.userId],ignore_index=True)))
item_ids = pd.Index(pd.unique(train_raw.movieId))
user2idx = {int(uid):k for k,uid in enumerate(user_ids)}
item2idx = {int(iid):k for k,iid in enumerate(item_ids)}
idx2item = item_ids.to_numpy()
num_users, num_items = len(user_ids),len(item_ids)

def encode_frame(df):
    out = df.copy()
    out['u'] = out.userId.map(user2idx)
    out['i'] = out.movieId.map(item2idx)
    if out[['u','i']].isna().any().any():
        raise ValueError('Unrecognized user/item IDs')
    out[['u','i']] = out[['u','i']].astype(np.int64)
    return out

def make_train_sets(df, threshold):
    frame = encode_frame(df)
    rating_group = frame.groupby('u')['rating']
    mu = rating_group.transform('mean')
    std = rating_group.transform('std').replace(0.0,np.nan)
    frame['rating_z'] = ((frame.rating-mu)/std).fillna(0.0)
    positive_pairs = frame.loc[frame.rating_z.ge(threshold),['u','i']].to_numpy(dtype=np.int64)
    observed = {int(u):set(g.i.astype(int).tolist()) for u,g in frame.groupby('u')}
    popularity = np.bincount(frame.i.to_numpy(),minlength=num_items).astype(np.float64)
    return frame,positive_pairs,observed,popularity

train_encoded, train_pairs, observed_train, popularity_train = make_train_sets(train_part,Z_THRESHOLD)
val_enc = encode_frame(val_relevant)
val_truth = {int(u):set(g.i.astype(int).tolist()) for u,g in val_enc.groupby('u')}
print('Positive pairs:',len(train_pairs),'Positive users:',len(np.unique(train_pairs[:,0])))
print('All users:',num_users,'All items:',num_items)
assert len(train_pairs)>0


Positive pairs: 16050 Positive users: 587
All users: 610 All items: 8237


## 4. NeuMF model: separate GMF and MLP embedding tables

GMF: elementwise product `p_u^G ⊙ q_i^G`. MLP: concatenate `p_u^M || q_i^M`, pass through fully connected/ReLU layers. Concatenate GMF and MLP representations, predict one logit. `USE_ITEM_BIAS=True` adds a learnable item bias as a *documented extension*. We deliberately avoid an additive user bias: it is constant across candidate items for the same user, so it cannot change a user's Top-K ordering.

In [55]:
class NeuMF(nn.Module):
    def __init__(self,n_users,n_items,gmf_dim=32,mlp_emb_dim=32,mlp_hidden=(32,16,8),use_item_bias=True):
        super().__init__()
        self.gmf_user = nn.Embedding(n_users,gmf_dim)
        self.gmf_item = nn.Embedding(n_items,gmf_dim)
        self.mlp_user = nn.Embedding(n_users,mlp_emb_dim)
        self.mlp_item = nn.Embedding(n_items,mlp_emb_dim)
        layers=[]
        previous = 2*mlp_emb_dim
        for hidden in mlp_hidden:
            layers.extend([nn.Linear(previous,hidden),nn.ReLU()])
            previous=hidden
        self.mlp=nn.Sequential(*layers)
        self.predictor=nn.Linear(gmf_dim+previous,1)
        self.item_bias = nn.Embedding(n_items,1) if use_item_bias else None
        for emb in (self.gmf_user,self.gmf_item,self.mlp_user,self.mlp_item):
            nn.init.normal_(emb.weight,std=0.01)
        if self.item_bias is not None:nn.init.zeros_(self.item_bias.weight)
        nn.init.xavier_uniform_(self.predictor.weight)
        nn.init.zeros_(self.predictor.bias)

    def forward(self,u,i):
        gmf = self.gmf_user(u)*self.gmf_item(i)
        mlp_input = torch.cat([self.mlp_user(u),self.mlp_item(i)],dim=-1)
        mlp = self.mlp(mlp_input)
        logit = self.predictor(torch.cat([gmf,mlp],dim=-1)).squeeze(-1)
        if self.item_bias is not None:
            logit = logit + self.item_bias(i).squeeze(-1)
        return logit

def make_model():
    return NeuMF(num_users,num_items,GMF_DIM,MLP_EMBED_DIM,MLP_HIDDEN,USE_ITEM_BIAS).to(DEVICE)

model=make_model()
print(model)
print('Trainable parameters:',sum(x.numel() for x in model.parameters()))


NeuMF(
  (gmf_user): Embedding(610, 32)
  (gmf_item): Embedding(8237, 32)
  (mlp_user): Embedding(610, 32)
  (mlp_item): Embedding(8237, 32)
  (mlp): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=16, bias=True)
    (3): ReLU()
    (4): Linear(in_features=16, out_features=8, bias=True)
    (5): ReLU()
  )
  (predictor): Linear(in_features=40, out_features=1, bias=True)
  (item_bias): Embedding(8237, 1)
)
Trainable parameters: 577230


## 5. Negative sampling and pointwise BCE training

In [56]:
def get_negative_probs(pop,sampling,alpha=0.5,beta=0.75):
    n=len(pop)
    uniform=np.full(n,1.0/n)
    if sampling=='uniform':return uniform
    if sampling!='popularity_mixed':raise ValueError(sampling)
    powered=np.power(pop+1e-6,beta)
    powered/=powered.sum()
    return (1-alpha)*uniform+alpha*powered

def sample_negatives(users,observed,probs,rng):
    # Rejection sampling excludes ALL observed training movies (including low ratings).
    out=rng.choice(len(probs),size=len(users),p=probs)
    bad=np.fromiter((int(i) in observed.get(int(u),set()) for u,i in zip(users,out)),dtype=bool,count=len(users))
    while bad.any():
        out[bad]=rng.choice(len(probs),size=int(bad.sum()),p=probs)
        bad=np.fromiter((int(i) in observed.get(int(u),set()) for u,i in zip(users,out)),dtype=bool,count=len(users))
    return out.astype(np.int64)

def train_one_epoch(model,optimizer,positive_pairs,observed,probs,seed):
    model.train()
    rng=np.random.default_rng(seed)
    # Construct pointwise positive (label=1) and sampled negatives (label=0) each epoch.
    p_u,p_i=positive_pairs[:,0],positive_pairs[:,1]
    n_u=np.repeat(p_u,NUM_NEGATIVES)
    n_i=sample_negatives(n_u,observed,probs,rng)
    uu=np.concatenate([p_u,n_u])
    ii=np.concatenate([p_i,n_i])
    yy=np.concatenate([np.ones(len(p_u),dtype=np.float32),np.zeros(len(n_u),dtype=np.float32)])
    perm=rng.permutation(len(uu))
    total=0.0
    for begin in range(0,len(perm),BATCH_SIZE):
        ix=perm[begin:begin+BATCH_SIZE]
        u=torch.as_tensor(uu[ix],dtype=torch.long,device=DEVICE)
        i=torch.as_tensor(ii[ix],dtype=torch.long,device=DEVICE)
        y=torch.as_tensor(yy[ix],dtype=torch.float32,device=DEVICE)
        loss=F.binary_cross_entropy_with_logits(model(u,i),y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        total+=float(loss.item())*len(ix)
    return total/len(uu)

train_probs=get_negative_probs(popularity_train,NEG_SAMPLING,POPULARITY_ALPHA,POPULARITY_BETA)


## 6. Full-catalog Top-20 & validation NDCG

No 99-negative sampled evaluation: score **every catalog item** per test user. Mask only movies observed in the training split. Validation truth = movies with held-out `rating >= 4`; evaluate test-target users with at least one relevant movie (matching Q1 BPR v2). `sigmoid(logit)` is unnecessary for ranking because it is monotonic.

In [57]:
@torch.no_grad()
def topk_recommend(model,user_indices,observed,k=20,train_popularity=None,user_batch=8,item_batch=2048):
    model.eval()
    users=np.asarray(user_indices,dtype=np.int64)
    output=np.empty((len(users),k),dtype=np.int64)
    pop_fallback=np.argsort(-train_popularity,kind='stable') if train_popularity is not None else None
    # Users with no positive training pairs have untrained embeddings; fallback to popularity.
    for start in range(0,len(users),user_batch):
        block=users[start:start+user_batch]
        scores=np.empty((len(block),num_items),dtype=np.float32)
        for j0 in range(0,num_items,item_batch):
            j1=min(num_items,j0+item_batch)
            bi=np.arange(j0,j1,dtype=np.int64)
            uu=torch.as_tensor(np.repeat(block,len(bi)),dtype=torch.long,device=DEVICE)
            ii=torch.as_tensor(np.tile(bi,len(block)),dtype=torch.long,device=DEVICE)
            logits=model(uu,ii).view(len(block),len(bi))
            scores[:,j0:j1]=logits.cpu().numpy()
        for row,u in enumerate(block):
            seen=observed.get(int(u),set())
            if pop_fallback is not None and int(u) not in trained_positive_users:
                scores[row]=train_popularity.astype(np.float32)
            if seen: scores[row,list(seen)]=-np.inf
            top=np.argpartition(-scores[row],kth=k-1)[:k]
            output[start+row]=top[np.argsort(-scores[row,top],kind='stable')]
    return output

def evaluate(model,truth,observed,pop):
    eligible=np.asarray(sorted(truth),dtype=np.int64)
    if len(eligible)==0: raise ValueError('No eligible validation users')
    recs=topk_recommend(model,eligible,observed,k=K,train_popularity=pop,
                        user_batch=EVAL_USER_BATCH,item_batch=EVAL_ITEM_BATCH)
    ndcgs=[];hit_rates=[]
    for u,ranked in zip(eligible,recs):
        relevant=truth[int(u)]
        hits=np.asarray([int(int(i) in relevant) for i in ranked])
        dcg=(hits/np.log2(np.arange(2,K+2))).sum()
        ideal=(1/np.log2(np.arange(2,min(K,len(relevant))+2))).sum()
        ndcgs.append(dcg/ideal)
        hit_rates.append(float(hits.any()))
    return float(np.mean(ndcgs)),float(np.mean(hit_rates))

trained_positive_users=set(map(int,np.unique(train_pairs[:,0])))


## 7. Train with early stopping on **validation NDCG@20**

In [58]:
model=make_model()
optimizer=torch.optim.Adam(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
best_ndcg=-1.0
best_epoch=0
best_state=None
no_improve=0
history=[]
for epoch in range(1,MAX_EPOCHS+1):
    loss=train_one_epoch(model,optimizer,train_pairs,observed_train,train_probs,SEED+epoch)
    if epoch%EVAL_EVERY==0:
        ndcg,hit=evaluate(model,val_truth,observed_train,popularity_train)
        history.append({'epoch':epoch,'train_loss':loss,'val_ndcg20':ndcg,'val_hit20':hit})
        print(f'Epoch {epoch:03d} | Train BCE {loss:.5f} | Val NDCG@20 {ndcg:.5f} | Hit@20 {hit:.5f}')
        if ndcg>best_ndcg+1e-7:
            best_ndcg,best_epoch=ndcg,epoch
            best_state=copy.deepcopy(model.state_dict())
            no_improve=0
        else:
            no_improve+=1
        if no_improve>=PATIENCE:
            print('Early stopping.')
            break
history_df=pd.DataFrame(history)
history_path=OUTPUT_DIR/f'q2_NeuMF_history_{EXP_NAME}.csv'

history_df.to_csv(history_path,index=False)
assert best_state is not None
model.load_state_dict(best_state)
print('BEST:',best_epoch,'Val NDCG@20:',round(best_ndcg,6),'History:',history_path)
display(history_df.tail())

Epoch 001 | Train BCE 0.68848 | Val NDCG@20 0.09265 | Hit@20 0.45217
Epoch 002 | Train BCE 0.64861 | Val NDCG@20 0.09184 | Hit@20 0.44000
Epoch 003 | Train BCE 0.52593 | Val NDCG@20 0.09300 | Hit@20 0.44522
Epoch 004 | Train BCE 0.41571 | Val NDCG@20 0.09326 | Hit@20 0.44174
Epoch 005 | Train BCE 0.36963 | Val NDCG@20 0.09387 | Hit@20 0.44696
Epoch 006 | Train BCE 0.35466 | Val NDCG@20 0.09386 | Hit@20 0.44348
Epoch 007 | Train BCE 0.34736 | Val NDCG@20 0.09247 | Hit@20 0.45217
Epoch 008 | Train BCE 0.34028 | Val NDCG@20 0.09223 | Hit@20 0.45913
Epoch 009 | Train BCE 0.33496 | Val NDCG@20 0.09357 | Hit@20 0.46609
Epoch 010 | Train BCE 0.32834 | Val NDCG@20 0.09381 | Hit@20 0.45913
Epoch 011 | Train BCE 0.32060 | Val NDCG@20 0.09360 | Hit@20 0.46261
Epoch 012 | Train BCE 0.31553 | Val NDCG@20 0.09476 | Hit@20 0.46957
Epoch 013 | Train BCE 0.30799 | Val NDCG@20 0.09696 | Hit@20 0.46957
Epoch 014 | Train BCE 0.30097 | Val NDCG@20 0.09933 | Hit@20 0.47304
Epoch 015 | Train BCE 0.29423 | Va

,epoch,train_loss,val_ndcg20,val_hit20
42,43,0.139866,0.112324,0.554783
43,44,0.137009,0.112827,0.549565
44,45,0.133710,0.113998,0.568696
45,46,0.131670,0.115816,0.572174
46,47,0.127245,0.111455,0.554783


## 8. Refit on full `train.csv` with selected epoch count

Compute user Z-score again from **all** train.csv rows for the final model. Reinitialize the network with the same seed, train for `best_epoch` epochs; final checkpoint can differ from validation checkpoint because there are more positives. Log the choice in your report.

In [59]:
full_encoded,full_pairs,observed_full,popularity_full=make_train_sets(train_raw,Z_THRESHOLD)
full_probs=get_negative_probs(popularity_full,NEG_SAMPLING,POPULARITY_ALPHA,POPULARITY_BETA)
trained_positive_users=set(map(int,np.unique(full_pairs[:,0])))
print('Full-training positives:',len(full_pairs))
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
final_model=make_model()
final_optimizer=torch.optim.Adam(final_model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
for epoch in range(1,best_epoch+1):
    loss=train_one_epoch(final_model,final_optimizer,full_pairs,observed_full,full_probs,SEED+epoch)
    if epoch==1 or epoch%5==0 or epoch==best_epoch:
        print(f'Final epoch {epoch:03d}/{best_epoch} | train BCE={loss:.5f}')


Full-training positives: 20057
Final epoch 001/39 | train BCE=0.61688
Final epoch 005/39 | train BCE=0.35113
Final epoch 010/39 | train BCE=0.30569
Final epoch 015/39 | train BCE=0.25247
Final epoch 020/39 | train BCE=0.21268
Final epoch 025/39 | train BCE=0.18249
Final epoch 030/39 | train BCE=0.15608
Final epoch 035/39 | train BCE=0.13640
Final epoch 039/39 | train BCE=0.12303


## 9. Generate and validate Kaggle submission (20 movie IDs per user)

In [60]:
test_encoded=test.userId.map(user2idx).to_numpy(dtype=np.int64)
recommendations_idx=topk_recommend(final_model,test_encoded,observed_full,k=K,
                                    train_popularity=popularity_full,user_batch=EVAL_USER_BATCH,
                                    item_batch=EVAL_ITEM_BATCH)
recommendations_movie_ids=idx2item[recommendations_idx]
submission=test[['id']].copy()
submission['recommendations']=[' '.join(map(str,row)) for row in recommendations_movie_ids]
assert submission.shape==(len(test),2)
assert submission.id.tolist()==sample.id.tolist()
assert submission.recommendations.apply(lambda s: len(s.split())==K and len(set(s.split()))==K).all()
assert all(not(set(map(int,row))&set(train_raw.loc[train_raw.userId.eq(uid),'movieId']))
           for uid,row in zip(test.userId,recommendations_movie_ids)), 'Recommendation includes a train-observed movie.'
submission_path=OUTPUT_DIR/f'submission_q2_NeuMF_{EXP_NAME}.csv'

submission.to_csv(submission_path,index=False)
print('Saved:',submission_path,'Shape:',submission.shape)
display(submission.head())


Saved: /kaggle/working/outputs/submission_q2_NeuMF_exp5.csv Shape: (588, 2)


,id,recommendations
0,0,2344 5361 8819 6281 8574 5922 3527 7263 3949 8...
1,1,3878 7837 5650 6089 1555 166 4527 425 1716 106...
2,2,2344 5361 8574 5922 4480 2401 6281 1555 3016 4...
3,3,6972 9047 3713 2577 1879 290 7899 5532 6281 78...
4,4,4115 1037 2256 6281 8309 7455 7332 156 1746 35...


## 10. Experiment / report notes

- **Training:** pointwise BCEWithLogits with four dynamic negatives per positive; this follows the official NeuMF approach (as opposed to BPR loss).
- **Model:** GMF branch + independent MLP branch + final linear prediction; default `USE_ITEM_BIAS=True` is an **extension** to the basic architecture.
- **Rating:** normalized Z-score (`>= 0.5`) defines training positives. In validation, relevance is held fixed at raw rating `>= 4`, for comparability with Q1.
- **Time:** used in the temporal split only, not as a sequential recommender input.
- **Selection:** best validation NDCG@20, not minimum training BCE. Do not claim that this metric equals hidden Kaggle evaluation.
- **Recommended experiments:** (A) item bias on vs off; (B) `uniform` vs `popularity_mixed` negatives; (C) Z thresholds, only with the validation split and relevance held fixed.
- **Original source:** [He et al., NeuMF implementation](https://github.com/hexiangnan/neural_collaborative_filtering).